# 03. Model and Training

This notebook defines the entire model architecture (Encoders, Fusion, Heads, Symbolic Layer, LTN Predicates) and runs the training loop.

## 1. Imports and Configuration

In [1]:
import os
import yaml
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import numpy as np
from collections import defaultdict
import random
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

# We will import the dataset components from the python file for brevity, or assume they are defined.
import sys
sys.path.insert(0, os.path.abspath('../src'))
from dataset import load_and_preprocess, split_data, fit_scaler, build_dataloaders, TARGET_COLS

## 2. Encoders
Dual Image Encoder (EfficientNet + ViT) and Tabular Encoder.

In [2]:
"""
Encoders module - Image and Tabular feature extractors.
"""

import torch
import torch.nn as nn
import torchvision.models as models

class DualImageEncoder(nn.Module):
    def __init__(self, out_dim=256, eff_dropout=0.3, freeze_eff=4, freeze_vit_pct=0.75):
        super().__init__()

        # --- EfficientNet-B0: local texture, vegetation density, colour ---
        backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        features = list(backbone.features.children())
        for i, block in enumerate(features):
            if i < freeze_eff:
                for p in block.parameters():
                    p.requires_grad = False
        self.eff_backbone = backbone.features
        self.eff_pool     = nn.AdaptiveAvgPool2d(1)
        self.eff_proj     = nn.Sequential(
            nn.Dropout(eff_dropout),
            nn.Linear(1280, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.GELU()
        )

        # --- ViT-B/16: global semantics, pasture composition ---
        vit = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
        n_blocks = len(vit.encoder.layers)
        freeze_n = int(n_blocks * freeze_vit_pct)  # freeze ~75% of transformer blocks
        for i, block in enumerate(vit.encoder.layers):
            if i < freeze_n:
                for p in block.parameters():
                    p.requires_grad = False
        # Store ViT components separately so we can extract CLS token
        # (the full vit.forward() returns 1000-dim logits, not 768-dim features)
        self.vit_conv_proj   = vit.conv_proj
        self.vit_class_token = vit.class_token
        self.vit_pos_embed   = vit.encoder.pos_embedding
        self.vit_encoder_ln  = vit.encoder.ln
        self.vit_encoder_layers = vit.encoder.layers
        self.vit_seq_length  = vit.seq_length
        self.vit_hidden_dim  = vit.hidden_dim

        self.vit_proj = nn.Sequential(
            nn.Linear(768, out_dim),
            nn.BatchNorm1d(out_dim),
            nn.GELU()
        )

    def _vit_features(self, x):
        """Extract CLS token (768-dim) from ViT, bypassing the classification head."""
        # Patch embedding
        B = x.shape[0]
        x = self.vit_conv_proj(x)             # [B, hidden_dim, H', W']
        x = x.flatten(2).transpose(1, 2)      # [B, n_patches, hidden_dim]

        # Prepend CLS token
        cls_tokens = self.vit_class_token.expand(B, -1, -1)
        x = torch.cat([cls_tokens, x], dim=1) # [B, 1+n_patches, hidden_dim]

        # Add positional embedding
        x = x + self.vit_pos_embed

        # Transformer encoder blocks
        for layer in self.vit_encoder_layers:
            x = layer(x)

        x = self.vit_encoder_ln(x)
        return x[:, 0]                         # [B, 768] - CLS token only

    def forward(self, x):
        # EfficientNet branch
        f_eff = self.eff_backbone(x)            # [B, 1280, 7, 7]
        f_eff = self.eff_pool(f_eff).flatten(1) # [B, 1280]
        f_eff = self.eff_proj(f_eff)            # [B, 256]

        # ViT branch (CLS token)
        f_vit = self._vit_features(x)          # [B, 768]
        f_vit = self.vit_proj(f_vit)            # [B, 256]

        return torch.cat([f_eff, f_vit], dim=1)  # [B, 512]


class TabularEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        tab_cfg = cfg['model']['tabular_encoder']
        
        # Embeddings
        self.species_emb = nn.Embedding(15, tab_cfg['species_emb_dim']) 
        self.state_emb   = nn.Embedding(4, tab_cfg['state_emb_dim'])
        self.month_emb   = nn.Embedding(12, tab_cfg['month_emb_dim'])
        self.season_emb  = nn.Embedding(4, tab_cfg['season_emb_dim'])
        
        # Calculate total dimension
        cat_dim = (tab_cfg['species_emb_dim'] + 
                   tab_cfg['state_emb_dim'] + 
                   tab_cfg['month_emb_dim'] + 
                   tab_cfg['season_emb_dim'])
        num_dim = 2 # NDVI, Height
        in_dim = cat_dim + num_dim
        
        hidden_dim = tab_cfg['hidden_dim']
        dropout = tab_cfg['dropout']

        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )

    def forward(self, tab):
        cat = torch.cat([
            self.species_emb(tab['species']),
            self.state_emb(tab['state']),
            self.month_emb(tab['month']),
            self.season_emb(tab['season']),
            tab['numerical']                      # [NDVI, log1p_Height], normalized
        ], dim=1)
        return self.mlp(cat)   # [B, 128]


## 3. Fusion Module
Cross-Modal Attention with FiLM conditioning.

In [3]:
"""
Cross-Modal Attention Fusion Module.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

class CrossModalAttentionFusion(nn.Module):
    def __init__(self, img_dim=512, tab_dim=128, attn_dim=128, hidden_dim=256, dropout=0.3):
        super().__init__()
        # Cross-attention: image (visual) queries tabular context
        self.q = nn.Linear(img_dim, attn_dim)
        self.k = nn.Linear(tab_dim, attn_dim)
        self.v = nn.Linear(tab_dim, attn_dim)
        self.scale = attn_dim ** -0.5

        # FiLM-style conditioning: tabular generates γ, β for visual features
        self.film_gamma = nn.Linear(tab_dim, img_dim)
        self.film_beta  = nn.Linear(tab_dim, img_dim)

        # Fusion MLP
        fused_dim = img_dim + tab_dim + attn_dim  # 512+128+128 = 768
        self.mlp = nn.Sequential(
            nn.Linear(fused_dim, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU()
        )

    def forward(self, f_img, f_tab):
        # FiLM modulation: metadata conditions the visual features
        gamma = self.film_gamma(f_tab)                      # [B, 512]
        beta  = self.film_beta(f_tab)                       # [B, 512]
        f_img_cond = gamma * f_img + beta                   # [B, 512]

        # Cross-modal attention
        Q = self.q(f_img_cond).unsqueeze(1)                 # [B, 1, 128]
        K = self.k(f_tab).unsqueeze(1)                      # [B, 1, 128]
        V = self.v(f_tab).unsqueeze(1)                      # [B, 1, 128]
        attn_w = F.softmax(Q @ K.transpose(-2,-1) * self.scale, dim=-1)
        f_attn = (attn_w @ V).squeeze(1)                    # [B, 128]

        # Fuse all representations
        fused = torch.cat([f_img_cond, f_tab, f_attn], dim=1)  # [B, 768]
        return self.mlp(fused)                               # [B, 256] = Z


## 4. Heads & Symbolic Layer
Regression heads and the structural conservation layer (GDM = Green + Clover, Total = GDM + Dead).

In [4]:
"""
Regression heads module.
"""

import torch
import torch.nn as nn

class RegressionHead(nn.Module):
    def __init__(self, in_dim=256, hidden=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 64),
            nn.GELU(),
            nn.Linear(64, 1),
            nn.Softplus()    # Output >= 0 in log1p space (log1p(0) = 0)
        )

    def forward(self, z):
        return self.net(z).squeeze(-1)   # [B]


class CloverHead(nn.Module):
    """Two-stage: (1) binary presence gate, (2) conditional amount regression."""
    def __init__(self, in_dim=256, dropout=0.2):
        super().__init__()
        # Stage 1: Is clover present?
        self.presence_gate = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)        # logit: sigmoid(.) = P(clover > 0)
        )
        # Stage 2: How much clover? (predicted only when present)
        self.amount_net = nn.Sequential(
            nn.Linear(in_dim, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, 1),
            nn.Softplus()
        )

    def forward(self, z):
        p_present = torch.sigmoid(self.presence_gate(z))    # [B, 1] in (0,1)
        amount    = self.amount_net(z)                       # [B, 1] > 0
        return (p_present * amount).squeeze(-1)              # [B], gated prediction

    def presence_logit(self, z):
        return self.presence_gate(z).squeeze(-1)             # [B], for BCE loss


class HeteroscedasticHead(nn.Module):
    def __init__(self, in_dim=256, hidden=128):
        super().__init__()
        self.shared = nn.Sequential(nn.Linear(in_dim, hidden), nn.GELU())
        self.mu_head    = nn.Sequential(nn.Linear(hidden, 1), nn.Softplus())
        self.logvar_head = nn.Linear(hidden, 1)   # unconstrained

    def forward(self, z):
        h = self.shared(z)
        mu     = self.mu_head(h).squeeze(-1)       # [B]
        logvar = self.logvar_head(h).squeeze(-1)   # [B]
        return mu, logvar

"""
Symbolic Conservation Layer module.
"""

import torch
import torch.nn as nn

class SymbolicConservationLayer(nn.Module):
    """Derives GDM and Total from primary predictions in original (non-log) space.
    The additive conservation law is encoded here, not in the loss.
    CSR for P1 and P2 is 100% by construction.
    """
    def forward(self, green_log, dead_log, clover_log):
        # Back-transform primary predictions to original space
        green  = torch.expm1(green_log.clamp(min=0))    # [B]
        dead   = torch.expm1(dead_log.clamp(min=0))     # [B]
        clover = torch.expm1(clover_log.clamp(min=0))   # [B]

        # --- Biological Conservation Laws (exact by construction) ------------
        gdm   = green + clover            # Rule: GDM = Green + Clover
        total = gdm + dead                # Rule: Total = GDM + Dead

        return {
            # Log-space (for regression loss comparison with log-transformed targets)
            'Dry_Green_g':  green_log,
            'Dry_Dead_g':   dead_log,
            'Dry_Clover_g': clover_log,
            'GDM_g':        torch.log1p(gdm.clamp(min=0)),
            'Dry_Total_g':  torch.log1p(total.clamp(min=0)),
            # Original space (for LTN predicates P3-P11 and post-hoc analysis)
            '_green':  green,
            '_dead':   dead,
            '_clover': clover,
            '_gdm':    gdm,
            '_total':  total,
        }


## 5. LTN Predicates
Biological constraints implemented as differentiable fuzzy logic.

In [5]:
"""
LTN Predicates module - Pure PyTorch implementation of LTN-style fuzzy predicates.

All predicates use sigmoid-based fuzzy logic with temperature-controlled
sharpness (tau). The satisfiability of each predicate is a differentiable
scalar in [0, 1] that can be aggregated into an LTN loss.
"""

import torch
import torch.nn as nn
import numpy as np

class P3_Clover_Subset_GDM(nn.Module):
    def forward(self, clover, gdm, tau):
        margin = gdm - clover
        return torch.sigmoid(margin / tau)

class P4_Total_Is_Maximum(nn.Module):
    def forward(self, total, clover, dead, green, gdm, tau):
        others = torch.stack([clover, dead, green, gdm], dim=1)  # [B, 4]
        margins = total.unsqueeze(1) - others
        sats = torch.sigmoid(margins / tau)
        return sats.min(dim=1).values

class P5_NDVI_GDM_Monotonic(nn.Module):
    def forward(self, ndvi_i, ndvi_j, gdm_i, gdm_j, tau):
        agreement = (ndvi_i - ndvi_j) * (gdm_i - gdm_j)
        return torch.sigmoid(agreement / (tau * 10))

class P6_WA_Dead_Zero(nn.Module):
    def forward(self, dead, is_wa_mask, tau):
        if is_wa_mask.sum() == 0:
            return torch.tensor(1.0, device=dead.device)
        wa_dead = dead[is_wa_mask]
        return torch.sigmoid(-wa_dead / tau).mean()

class P7_CloverZeroSpecies(nn.Module):
    def forward(self, clover, is_clover_zero, tau):
        if is_clover_zero.sum() == 0:
            return torch.tensor(1.0, device=clover.device)
        return torch.sigmoid(-clover[is_clover_zero] / tau).mean()

class P8_PureCloverSpecies(nn.Module):
    def forward(self, dead, green, is_pure_clover, tau):
        if is_pure_clover.sum() == 0:
            return torch.tensor(1.0, device=dead.device)
        sat_dead  = torch.sigmoid(-dead[is_pure_clover]  / tau).mean()
        sat_green = torch.sigmoid(-green[is_pure_clover] / tau).mean()
        return torch.min(sat_dead, sat_green)

class P9_WinterDeadProportion(nn.Module):
    def forward(self, dead, total, is_winter, tau):
        if is_winter.sum() == 0:
            return torch.tensor(1.0, device=dead.device)
        prop = dead[is_winter] / total[is_winter].clamp(min=1e-6)
        return torch.sigmoid((prop - 0.24) / tau).mean()

class P10_SummerGreenProportion(nn.Module):
    def forward(self, green, total, is_summer, tau):
        if is_summer.sum() == 0:
            return torch.tensor(1.0, device=green.device)
        prop = green[is_summer] / total[is_summer].clamp(min=1e-6)
        return torch.sigmoid((prop - 0.657) / tau).mean()

class P11_Height_GDM_Monotonic(nn.Module):
    def forward(self, log_height_i, log_height_j, gdm_i, gdm_j, tau):
        agreement = (log_height_i - log_height_j) * (gdm_i - gdm_j)
        return torch.sigmoid(agreement / (tau * 10))


def get_tau_dict(cfg, epoch):
    """
    Computes current tau values based on cosine annealing schedule.
    """
    total_epochs = cfg['training']['epochs']
    tau_end = cfg['ltn']['tau']
    anneal_factor = cfg['ltn']['tau_anneal_factor']
    
    tau_start = {k: v * anneal_factor for k, v in tau_end.items()}
    
    progress = min(epoch / total_epochs, 1.0)
    current_tau = {}
    
    for category in tau_end.keys():
        t = tau_end[category]
        t0 = tau_start[category]
        current_tau[category] = t + 0.5 * (t0 - t) * (1 + np.cos(np.pi * progress))
        
    return current_tau


## 6. Model Assembly
Tying everything together.

In [6]:
"""
Full BiomassLTNModel Assembly.
"""

import torch
import torch.nn as nn

class BiomassLTNModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        
        # Encoders
        enc_cfg = cfg['model']['img_encoder']
        self.img_encoder = DualImageEncoder(
            out_dim=enc_cfg['out_dim'],
            eff_dropout=enc_cfg['eff_dropout'],
            freeze_eff=enc_cfg['freeze_eff_blocks'],
            freeze_vit_pct=enc_cfg['freeze_vit_pct']
        )
        self.tab_encoder = TabularEncoder(cfg)
        
        # Fusion
        fus_cfg = cfg['model']['fusion']
        self.fusion = CrossModalAttentionFusion(
            img_dim=fus_cfg['img_dim'],
            tab_dim=fus_cfg['tab_dim'],
            attn_dim=fus_cfg['attn_dim'],
            hidden_dim=fus_cfg['hidden_dim'],
            dropout=fus_cfg['dropout']
        )
        
        # Heads
        head_cfg = cfg['model']['heads']
        hidden = head_cfg['hidden_dims'][0]
        dropout = head_cfg['dropout']
        
        # Z dimension is hidden_dim from fusion (256)
        self.green_head  = RegressionHead(256, hidden, dropout)
        self.dead_head   = RegressionHead(256, hidden, dropout)
        
        if head_cfg['use_clover_gate']:
            self.clover_head = CloverHead(256, dropout)
        else:
            self.clover_head = RegressionHead(256, hidden, dropout)
            
        self.symbolic = SymbolicConservationLayer()

        # LTN Predicates (P3-P11)
        self.P3  = P3_Clover_Subset_GDM()
        self.P4  = P4_Total_Is_Maximum()
        self.P5  = P5_NDVI_GDM_Monotonic()
        self.P6  = P6_WA_Dead_Zero()
        self.P7  = P7_CloverZeroSpecies()
        self.P8  = P8_PureCloverSpecies()
        self.P9  = P9_WinterDeadProportion()
        self.P10 = P10_SummerGreenProportion()
        self.P11 = P11_Height_GDM_Monotonic()

    def forward(self, images, tab_batch, tau_dict):
        # Encode
        f_vis = self.img_encoder(images)
        f_tab = self.tab_encoder(tab_batch)
        Z     = self.fusion(f_vis, f_tab)

        # Primary predictions (log1p space)
        y_hat_green_log  = self.green_head(Z)
        y_hat_dead_log   = self.dead_head(Z)
        y_hat_clover_log = self.clover_head(Z)
        
        if isinstance(self.clover_head, CloverHead):
            clover_logit = self.clover_head.presence_logit(Z)
        else:
            clover_logit = None

        # Symbolic derivation (original space and log space combined)
        derived = self.symbolic(y_hat_green_log, y_hat_dead_log, y_hat_clover_log)
        
        # derived contains all 5 log-space preds + 5 original-space preds (_green etc.)

        # LTN satisfiabilities
        orig = {k.lstrip('_'): derived[k] for k in derived if k.startswith('_')}
        sat_dict = self._compute_satisfiabilities(orig, tab_batch, tau_dict)

        return derived, sat_dict, clover_logit

    def _compute_satisfiabilities(self, orig, tab, tau_dict):
        ndvi = tab['numerical'][:, 0]   # scaled NDVI
        h    = tab['numerical'][:, 1]   # scaled log1p(Height)

        # Pairwise predicates: use random half-batch pairs for efficiency
        B = orig['green'].size(0)
        idx_i = torch.randperm(B)[:B//2]
        idx_j = torch.randperm(B)[:B//2]

        return {
            'P3':  self.P3(orig['clover'], orig['gdm'], tau_dict['hard']).mean(),
            'P4':  self.P4(orig['total'], orig['clover'],
                           orig['dead'], orig['green'], orig['gdm'], tau_dict['hard']).mean(),
            'P5':  self.P5(ndvi[idx_i], ndvi[idx_j],
                           orig['gdm'][idx_i], orig['gdm'][idx_j], tau_dict['very_soft']).mean(),
            'P6':  self.P6(orig['dead'], tab['is_wa'], tau_dict['near_exact']),
            'P7':  self.P7(orig['clover'], tab['is_clover_zero'], tau_dict['near_exact']),
            'P8':  self.P8(orig['dead'], orig['green'], tab['is_pure_clover'], tau_dict['near_exact']),
            'P9':  self.P9(orig['dead'], orig['total'], tab['is_winter'], tau_dict['soft']),
            'P10': self.P10(orig['green'], orig['total'], tab['is_summer'], tau_dict['soft']),
            'P11': self.P11(h[idx_i], h[idx_j],
                            orig['gdm'][idx_i], orig['gdm'][idx_j], tau_dict['very_soft']).mean(),
        }


## 7. Loss Functions
Huber regression loss + Gate BCE + LTN Satisfiability Aggregation.

In [7]:
"""
Loss functions.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

def regression_loss(preds, targets_log, target_cols, delta):
    """
    preds: dict of [B] log-space predictions (all 5 targets, including derived)
    targets_log: [B, 5] log-space targets
    delta: Huber transition point, calibrated for log1p space
    """
    total_loss = 0.0
    per_target = {}
    for i, name in enumerate(target_cols):
        y_pred = preds[name]
        y_true = targets_log[:, i]
        loss = F.huber_loss(y_pred, y_true, delta=delta, reduction='mean')
        per_target[name] = loss.item()
        total_loss += loss
    return total_loss, per_target

def gate_loss(clover_presence_logit, y_clover_log, species_is_clover_zero):
    """
    Binary cross-entropy on the presence gate.
    Ground truth: 0 for clover-zero species; (y_clover > 0) for others.
    """
    if clover_presence_logit is None:
        return torch.tensor(0.0, device=y_clover_log.device)
        
    y_present = (y_clover_log > 0).float()                  # [B]
    y_present[species_is_clover_zero] = 0.0                 # Override for known-zero species
    return F.binary_cross_entropy_with_logits(clover_presence_logit, y_present)

def compute_ltn_loss(sat_dict, predicate_weights):
    if not sat_dict:
        return torch.tensor(0.0)
        
    weighted_sats = torch.stack([
        predicate_weights[k] * (1.0 - v) for k, v in sat_dict.items()
    ])
    
    # Normalize weights so they sum to 1
    weight_sum = sum(predicate_weights.values())
    weighted_sats = weighted_sats / weight_sum

    # pMeanError with p=2 over weighted unsatisfied terms
    p = 2
    ltn_loss = (weighted_sats.pow(p).sum()).pow(1.0 / p)
    return ltn_loss


class TotalLoss(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.alpha      = cfg['loss']['alpha']        # Regression loss weight
        self.gamma_gate = cfg['loss']['gamma_gate']   # Clover gate BCE weight
        self.delta      = cfg['loss']['huber_delta']
        self.predicate_weights = cfg['ltn']['predicate_weights']

    def forward(self, preds, targets_log, target_cols, sat_dict,
                clover_logit, y_clover_log, species_is_cz,
                beta=0.5):
        
        L_reg, per_target = regression_loss(preds, targets_log, target_cols, self.delta)
        L_ltn  = compute_ltn_loss(sat_dict, self.predicate_weights)
        L_gate = gate_loss(clover_logit, y_clover_log, species_is_cz)

        total = self.alpha * L_reg + beta * L_ltn + self.gamma_gate * L_gate
        
        breakdown = {
            'L_reg': L_reg.item(), 
            'L_ltn': L_ltn.item() if isinstance(L_ltn, torch.Tensor) else 0.0, 
            'L_gate': L_gate.item() if isinstance(L_gate, torch.Tensor) else 0.0, 
            **per_target
        }
        
        return total, breakdown


def get_ltn_beta(epoch, cfg):
    """LTN loss weight: linear ramp from 0 -> beta_max over warmup epochs.
    Train regression first, then gradually introduce symbolic constraints.
    """
    warmup = cfg['loss']['ltn_beta_warmup']
    beta_max = cfg['loss']['beta_max']
    
    if epoch < warmup:
        return beta_max * (epoch / warmup)
    return beta_max


## 8. Training Loops
Train and Validation functions.

In [8]:
"""
Training loop and utilities.
"""

import torch
from collections import defaultdict
import numpy as np

def train_epoch(model, loader, optimizer, loss_fn, scaler, epoch, device, cfg):
    model.train()
    running = defaultdict(float)
    beta = get_ltn_beta(epoch, cfg)
    tau_dict = get_tau_dict(cfg, epoch)
    use_amp = cfg['training'].get('amp', False)

    for images, tab_d, targets in loader:
        images  = images.to(device)
        targets = targets.to(device)     # [B, 5] in log1p space
        tab_d   = {k: v.to(device) for k, v in tab_d.items()}

        optimizer.zero_grad()
        
        if use_amp and scaler is not None:
            with torch.cuda.amp.autocast():
                preds, sat_dict, clover_logit = model(images, tab_d, tau_dict)
                loss, breakdown = loss_fn(
                    preds, targets, TARGET_COLS, sat_dict,
                    clover_logit, targets[:, TARGET_COLS.index('Dry_Clover_g')],
                    tab_d['is_clover_zero'], beta=beta
                )
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg['training']['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
        else:
            preds, sat_dict, clover_logit = model(images, tab_d, tau_dict)
            loss, breakdown = loss_fn(
                preds, targets, TARGET_COLS, sat_dict,
                clover_logit, targets[:, TARGET_COLS.index('Dry_Clover_g')],
                tab_d['is_clover_zero'], beta=beta
            )
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=cfg['training']['grad_clip'])
            optimizer.step()

        for k, v in breakdown.items():
            running[k] += v

    return {k: v / len(loader) for k, v in running.items()}


def val_epoch(model, loader, loss_fn, epoch, device, cfg):
    model.eval()
    running = defaultdict(float)
    beta = get_ltn_beta(epoch, cfg)
    tau_dict = get_tau_dict(cfg, epoch)
    
    # Storage for evaluation metrics
    all_preds_orig = defaultdict(list)
    all_targets_orig = defaultdict(list)
    all_sat_dict = defaultdict(list)
    
    with torch.no_grad():
        for images, tab_d, targets in loader:
            images  = images.to(device)
            targets = targets.to(device)
            tab_d   = {k: v.to(device) for k, v in tab_d.items()}

            preds, sat_dict, clover_logit = model(images, tab_d, tau_dict)
            
            # Loss computation
            loss, breakdown = loss_fn(
                preds, targets, TARGET_COLS, sat_dict,
                clover_logit, targets[:, TARGET_COLS.index('Dry_Clover_g')],
                tab_d['is_clover_zero'], beta=beta
            )
            
            for k, v in breakdown.items():
                running[k] += v
                
            # Collect original-space predictions for metrics
            # Note: targets are in log1p space, need expm1
            # Map target column names to internal orig-space keys from SymbolicConservationLayer
            _COL_TO_ORIG = {
                'Dry_Clover_g': '_clover',
                'Dry_Dead_g':   '_dead',
                'Dry_Green_g':  '_green',
                'Dry_Total_g':  '_total',
                'GDM_g':        '_gdm',
            }
            for i, col in enumerate(TARGET_COLS):
                # Back-transform targets
                t_orig = torch.expm1(targets[:, i].clamp(min=0))
                all_targets_orig[col].append(t_orig.cpu())
                
                # Predictions (already available in orig space from symbolic layer)
                orig_key = _COL_TO_ORIG[col]
                all_preds_orig[col].append(preds[orig_key].cpu())
                
            # Collect satisfiabilities
            for k, v in sat_dict.items():
                # Some are scalars, some might be batch-wise. Average scalars.
                if v.dim() == 0:
                    all_sat_dict[k].append(v.item())
                else:
                    all_sat_dict[k].extend(v.cpu().tolist())

    metrics = {k: v / len(loader) for k, v in running.items()}
    
    # Compute RMSE in original space
    for col in TARGET_COLS:
        y_true = torch.cat(all_targets_orig[col])
        y_pred = torch.cat(all_preds_orig[col])
        rmse = torch.sqrt(torch.mean((y_pred - y_true)**2)).item()
        metrics[f'RMSE_orig_{col}'] = rmse
        
    # Average satisfiabilities
    for k in all_sat_dict:
        metrics[f'SAT_{k}'] = np.mean(all_sat_dict[k])
        
    return metrics, all_preds_orig, all_targets_orig


## 9. Load Configuration & Prepare Data
Load the YAML config, set seeds, build data loaders, and prepare the training pipeline.

In [9]:
# Load configuration
config_path = os.path.abspath('../config.yaml')
with open(config_path, 'r') as f:
    cfg = yaml.safe_load(f)

# Reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(cfg['data'].get('random_seed', 42))

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Output directories
base_dir = os.path.abspath('..')
outputs_dir = os.path.join(base_dir, 'outputs')
checkpoints_dir = os.path.join(outputs_dir, 'checkpoints')
metrics_dir = os.path.join(outputs_dir, 'metrics')
os.makedirs(checkpoints_dir, exist_ok=True)
os.makedirs(metrics_dir, exist_ok=True)

print("Configuration loaded successfully.")
print(f"Epochs: {cfg['training']['epochs']}, Patience: {cfg['training']['patience']}")

Using device: cpu
Configuration loaded successfully.
Epochs: 15, Patience: 15


In [10]:
# Load and preprocess data
print("Loading and preprocessing data...")
csv_path = os.path.join(base_dir, cfg['data']['train_csv'])
img_root = os.path.join(base_dir, cfg['data']['img_root'])

df_wide, label_encoders = load_and_preprocess(csv_path, img_root, cfg)
train_df, val_df = split_data(df_wide, cfg)
train_df, val_df, scaler = fit_scaler(train_df, val_df, outputs_dir)

print(f"Train samples: {len(train_df)}, Val samples: {len(val_df)}")

# Build initial data loaders
train_loader, val_loader = build_dataloaders(train_df, val_df, cfg, epoch=0)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

Loading and preprocessing data...
Train samples: 285, Val samples: 72
Train batches: 17, Val batches: 5


c:\Users\Faaiz\Desktop\neuro-symbolic ai\src\dataset.py:316: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  weights[high_biomass] = 0.1 + 0.9 * progress


## 10. Instantiate Model, Optimizer & Scheduler

In [11]:
# Build model
print("Building model...")
model = BiomassLTNModel(cfg).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Loss function
loss_fn = TotalLoss(cfg).to(device)

# Optimizer with differential learning rates
backbone_params = list(model.img_encoder.parameters())
other_params = [p for n, p in model.named_parameters() if not n.startswith('img_encoder.')]

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': cfg['training']['optimizer']['lr_backbone'], 
     'weight_decay': cfg['training']['optimizer']['weight_decay_backbone']},
    {'params': other_params, 'lr': cfg['training']['optimizer']['lr_other'], 
     'weight_decay': cfg['training']['optimizer']['weight_decay_other']},
])

# Scheduler: Linear warmup -> Cosine annealing
warmup = LinearLR(optimizer, start_factor=0.1, total_iters=cfg['training']['scheduler']['warmup_epochs'])
cosine = CosineAnnealingLR(
    optimizer, 
    T_max=cfg['training']['scheduler']['T_max'] - cfg['training']['scheduler']['warmup_epochs'], 
    eta_min=cfg['training']['scheduler']['eta_min']
)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[cfg['training']['scheduler']['warmup_epochs']])

# AMP Scaler (only used with CUDA)
use_amp = cfg['training'].get('amp', False) and torch.cuda.is_available()
amp_scaler = torch.cuda.amp.GradScaler() if use_amp else None
print(f"AMP enabled: {use_amp}")

Building model...
Total parameters: 91,253,728
Trainable parameters: 27,397,150
AMP enabled: False


## 11. Run Training
Full training loop with curriculum learning, early stopping, and checkpointing.

In [12]:
from evaluate import full_evaluation_report

epochs = cfg['training']['epochs']
patience = cfg['training']['patience']
best_val_rmse = float('inf')
epochs_without_improve = 0

print(f"Starting training for {epochs} epochs...")
print("=" * 70)

for epoch in range(epochs):
    print(f"\n--- Epoch {epoch+1}/{epochs} ---")
    
    # Rebuild train loader if curriculum is active and in warmup phase
    curriculum_cfg = cfg['training'].get('curriculum', {})
    if curriculum_cfg.get('enabled', True) and epoch < curriculum_cfg.get('warmup_epochs', 30):
        train_loader, _ = build_dataloaders(train_df, val_df, cfg, epoch=epoch)

    # Train
    train_metrics = train_epoch(model, train_loader, optimizer, loss_fn, amp_scaler, epoch, device, cfg)
    
    # Validate
    val_metrics, all_preds, all_targets = val_epoch(model, val_loader, loss_fn, epoch, device, cfg)
    
    scheduler.step()

    # Print key metrics
    train_loss = train_metrics['L_reg'] + train_metrics.get('L_ltn', 0)
    val_loss = val_metrics['L_reg'] + val_metrics.get('L_ltn', 0)
    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    # Track mean primary target RMSE
    primary_targets = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g']
    mean_val_rmse = np.mean([val_metrics[f'RMSE_orig_{t}'] for t in primary_targets])
    print(f"Mean Val RMSE (Original Space): {mean_val_rmse:.4f}")
    
    # Print satisfiability metrics
    sat_keys = [k for k in val_metrics if k.startswith('SAT_')]
    if sat_keys:
        sat_str = " | ".join([f"{k}: {val_metrics[k]:.3f}" for k in sat_keys])
        print(f"Satisfiabilities: {sat_str}")

    # Checkpoint logic
    if mean_val_rmse < best_val_rmse:
        best_val_rmse = mean_val_rmse
        epochs_without_improve = 0
        
        torch.save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_metrics': val_metrics,
            'config': cfg
        }, os.path.join(checkpoints_dir, 'best_model.pt'))
        
        # Save evaluation report for the best model
        full_evaluation_report(val_loader, all_preds, all_targets, model, device, metrics_dir)
        print(">> Best model saved!")
    else:
        epochs_without_improve += 1
        print(f"   No improvement for {epochs_without_improve}/{patience} epochs")
        if epochs_without_improve >= patience:
            print(f"\nEarly stopping triggered after {epoch+1} epochs.")
            break

# Save last model
torch.save({
    'epoch': epoch,
    'model_state': model.state_dict(),
    'optimizer_state': optimizer.state_dict(),
    'val_metrics': val_metrics,
    'config': cfg
}, os.path.join(checkpoints_dir, 'last_model.pt'))

print("\n" + "=" * 70)
print(f"Training completed! Best Val RMSE: {best_val_rmse:.4f}")
print(f"Best model saved to: {checkpoints_dir}/best_model.pt")
print(f"Evaluation report saved to: {metrics_dir}/")

Starting training for 15 epochs...

--- Epoch 1/15 ---
Train Loss: 9.9172 | Val Loss: 11.2219
Mean Val RMSE (Original Space): 22.7411
Satisfiabilities: SAT_P3: 0.533 | SAT_P4: 0.535 | SAT_P5: 0.500 | SAT_P6: 0.366 | SAT_P7: 0.443 | SAT_P8: 0.746 | SAT_P9: 0.502 | SAT_P10: 0.698 | SAT_P11: 0.500
>> Best model saved!

--- Epoch 2/15 ---
Train Loss: 8.8871 | Val Loss: 7.9413
Mean Val RMSE (Original Space): 22.2417
Satisfiabilities: SAT_P3: 0.569 | SAT_P4: 0.558 | SAT_P5: 0.500 | SAT_P6: 0.286 | SAT_P7: 0.383 | SAT_P8: 0.707 | SAT_P9: 0.502 | SAT_P10: 0.698 | SAT_P11: 0.500
>> Best model saved!

--- Epoch 3/15 ---
Train Loss: 4.1552 | Val Loss: 2.5737
Mean Val RMSE (Original Space): 17.5826
Satisfiabilities: SAT_P3: 0.960 | SAT_P4: 0.726 | SAT_P5: 0.500 | SAT_P6: 0.024 | SAT_P7: 0.161 | SAT_P8: 0.600 | SAT_P9: 0.500 | SAT_P10: 0.700 | SAT_P11: 0.501
>> Best model saved!

--- Epoch 4/15 ---
Train Loss: 2.3955 | Val Loss: 2.1459
Mean Val RMSE (Original Space): 17.3901
Satisfiabilities: SAT_P

## 12. Post-Training Summary
View the evaluation report generated during training.

In [13]:
summary_file = os.path.join(metrics_dir, 'evaluation_summary.txt')
if os.path.exists(summary_file):
    with open(summary_file, 'r') as f:
        print(f.read())
else:
    print('Evaluation summary not yet generated. Run training first.')

=== Biomass Prediction Evaluation ===

--- Regression Metrics (Original Space) ---

Dry_Clover_g:
  RMSE: 3.4761
  MAE: 2.0906
  R2: 0.9027
  MAPE: 52.2953

Dry_Dead_g:
  RMSE: 11.4135
  MAE: 6.0988
  R2: 0.2915
  MAPE: 70.5210

Dry_Green_g:
  RMSE: 14.3347
  MAE: 7.7796
  R2: 0.7365
  MAPE: 47.4498

Dry_Total_g:
  RMSE: 19.3328
  MAE: 11.6411
  R2: 0.6028
  MAPE: 28.1798

GDM_g:
  RMSE: 14.5865
  MAE: 8.4839
  R2: 0.6861
  MAPE: 26.7705

--- Constraint Satisfaction Rates ---
CSR1_GDM_Conservation: 1.0000
CSR2_Total_Conservation: 1.0000
CSR3_Clover_LTE_GDM: 1.0000
CSR4_Total_GTE_Max: 1.0000
Symbolic_Drift_g: 0.0000

